<a href="https://colab.research.google.com/github/ShadiFarzankia/Human-Value-Detection/blob/master/Human_Value_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Importing the Libararies

In [2]:
!pip install -q transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 65.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 98.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.1 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
from sklearn import metrics
import transformers
import torch
import torch.nn as nn
import shutil
import sys
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [4]:
# Random seed to repeat experiments.
RANDOM_SEED = 42
transformers.set_seed(RANDOM_SEED)

### Loading the Data

In [6]:
#loading the data into variables
train_args = pd.read_csv("arguments-training.tsv",delimiter='\t')
valid_args = pd.read_csv("arguments-validation.tsv",delimiter='\t')
test_args = pd.read_csv("arguments-test.tsv",delimiter='\t')

train_labels = pd.read_csv("labels-training.tsv",delimiter='\t')
valid_labels = pd.read_csv("labels-validation.tsv",delimiter='\t')
test_labels = pd.read_csv("labels-test.tsv",delimiter='\t')

In [7]:
#checking the size of data
print("The size of train_args:", train_args.shape, "  The size of train_labes:", train_labels.shape)

print("The size of valid_args:", valid_args.shape, "  The size of valid_labes:", valid_labels.shape)

print("The size of test_args:", test_args.shape, "  The size of test_labes:", test_labels.shape)

The size of train_args: (5393, 4)   The size of train_labes: (5393, 21)
The size of valid_args: (1896, 4)   The size of valid_labes: (1896, 21)
The size of test_args: (1576, 4)   The size of test_labes: (1576, 21)


In [8]:
print(train_args.head())

  Argument ID                                   Conclusion       Stance  \
0      A01002                  We should ban human cloning  in favor of   
1      A01005                      We should ban fast food  in favor of   
2      A01006  We should end the use of economic sanctions      against   
3      A01007         We should abolish capital punishment      against   
4      A01008                We should ban factory farming      against   

                                             Premise  
0  we should ban human cloning as it will only ca...  
1  fast food should be banned because it is reall...  
2  sometimes economic sanctions are the only thin...  
3  capital punishment is sometimes the only optio...  
4  factory farming allows for the production of c...  


In [9]:
print(train_labels.head())

  Argument ID  Self-direction: thought  Self-direction: action  Stimulation  \
0      A01002                        0                       0            0   
1      A01005                        0                       0            0   
2      A01006                        0                       0            0   
3      A01007                        0                       0            0   
4      A01008                        0                       0            0   

   Hedonism  Achievement  Power: dominance  Power: resources  Face  \
0         0            0                 0                 0     0   
1         0            0                 0                 0     0   
2         0            0                 1                 0     0   
3         0            0                 0                 0     0   
4         0            0                 0                 0     0   

   Security: personal  ...  Tradition  Conformity: rules  \
0                   0  ...          0       

In [10]:
#meging the arguments and labels
training_full_data = pd.merge(train_args, train_labels, on='Argument ID')

In [11]:
print(training_full_data.head())

  Argument ID                                   Conclusion       Stance  \
0      A01002                  We should ban human cloning  in favor of   
1      A01005                      We should ban fast food  in favor of   
2      A01006  We should end the use of economic sanctions      against   
3      A01007         We should abolish capital punishment      against   
4      A01008                We should ban factory farming      against   

                                             Premise  Self-direction: thought  \
0  we should ban human cloning as it will only ca...                        0   
1  fast food should be banned because it is reall...                        0   
2  sometimes economic sanctions are the only thin...                        0   
3  capital punishment is sometimes the only optio...                        0   
4  factory farming allows for the production of c...                        0   

   Self-direction: action  Stimulation  Hedonism  Achievement 

In [12]:
#becuase of optimality, we join the columns "conclusion", "stance" , and "premise"
training_full_data['Context'] = training_full_data['Conclusion']+". " + training_full_data['Stance'] + ". " + training_full_data['Premise']
training_full_data.drop(labels=['Argument ID', 'Conclusion', 'Stance', 'Premise'], axis=1, inplace=True)

In [13]:
training_full_data.columns

Index(['Self-direction: thought', 'Self-direction: action', 'Stimulation',
       'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources',
       'Face', 'Security: personal', 'Security: societal', 'Tradition',
       'Conformity: rules', 'Conformity: interpersonal', 'Humility',
       'Benevolence: caring', 'Benevolence: dependability',
       'Universalism: concern', 'Universalism: nature',
       'Universalism: tolerance', 'Universalism: objectivity', 'Context'],
      dtype='object')

In [14]:
print(training_full_data.head())

   Self-direction: thought  Self-direction: action  Stimulation  Hedonism  \
0                        0                       0            0         0   
1                        0                       0            0         0   
2                        0                       0            0         0   
3                        0                       0            0         0   
4                        0                       0            0         0   

   Achievement  Power: dominance  Power: resources  Face  Security: personal  \
0            0                 0                 0     0                   0   
1            0                 0                 0     0                   1   
2            0                 1                 0     0                   0   
3            0                 0                 0     0                   0   
4            0                 0                 0     0                   1   

   Security: societal  ...  Conformity: rules  Conformit

In [15]:
#Bring the last column to the first
train_df = training_full_data[['Context', 'Self-direction: thought', 'Self-direction: action', 'Stimulation',
       'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources',
       'Face', 'Security: personal', 'Security: societal', 'Tradition',
       'Conformity: rules', 'Conformity: interpersonal', 'Humility',
       'Benevolence: caring', 'Benevolence: dependability',
       'Universalism: concern', 'Universalism: nature',
       'Universalism: tolerance', 'Universalism: objectivity']]

In [16]:
print(train_df.head())

                                             Context  Self-direction: thought  \
0  We should ban human cloning. in favor of. we s...                        0   
1  We should ban fast food. in favor of. fast foo...                        0   
2  We should end the use of economic sanctions. a...                        0   
3  We should abolish capital punishment. against....                        0   
4  We should ban factory farming. against. factor...                        0   

   Self-direction: action  Stimulation  Hedonism  Achievement  \
0                       0            0         0            0   
1                       0            0         0            0   
2                       0            0         0            0   
3                       0            0         0            0   
4                       0            0         0            0   

   Power: dominance  Power: resources  Face  Security: personal  ...  \
0                 0                 0     0       

In [17]:
target_list = ['Self-direction: thought', 'Self-direction: action', 'Stimulation',
       'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources',
       'Face', 'Security: personal', 'Security: societal', 'Tradition',
       'Conformity: rules', 'Conformity: interpersonal', 'Humility',
       'Benevolence: caring', 'Benevolence: dependability',
       'Universalism: concern', 'Universalism: nature',
       'Universalism: tolerance', 'Universalism: objectivity']

### Defing the Model


In [18]:
from transformers import BertTokenizer, BertModel

In [19]:
MODEL_NAME = 'bert-base-uncased'

MAX_LEN = 256
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 1e-05
OUT_CHANNELS = 768 if "base" in  MODEL_NAME else 1024

In [20]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [21]:
class CustomDataset(torch.utils.data.Dataset):

  def __init__(self, df, tokenizer, max_len):
    self.df = df
    self.tokenizer = tokenizer
    self.max_len = max_len
    self.text = self.df['Context']
    self.targets = self.df[target_list].values

  def __len__(self):
    return len(self.text)

  # the function to get texts one by one
  def __getitem__(self, index):
    # to ensure that our text doesn't have any garbage at all
    text = str(self.text[index])
    text = "".join(text.split())

    inputs = self.tokenizer.encode_plus(
        text,
        None,
        add_special_tokens = True,
        max_length = self.max_len,
        padding = 'max_length',
        return_token_type_ids = True,
        truncation = True,
        return_attention_mask = True,
        return_tensors = 'pt'
    )

    #returns the dictionary
    return {
        'input_ids': inputs['input_ids'].flatten(), #[1, 512] => [512]
        'attention_mask': inputs['attention_mask'].flatten(),
        'token_type_ids': inputs['token_type_ids'].flatten(),
        'targets': torch.FloatTensor(self.targets[index])
    }

In [22]:
train_size = 0.8
train_df = train_df.sample(frac=train_size, random_state=200).reset_index(drop=True)# we rest the index so the index won't get messed up after resampling the rows
val_df = train_df.drop(train_df.index).reset_index(drop=True)

In [23]:
training_full_data[target_list].values

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 1, 0, ..., 0, 0, 0]])

In [24]:
train_dataset = CustomDataset(train_df, tokenizer, MAX_LEN)
valid_dataset = CustomDataset(val_df, tokenizer, MAX_LEN)

In [25]:
# Creating Data Loader
train_data_loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle = True,
    batch_size = TRAIN_BATCH_SIZE,
    num_workers = 0, # use all the gpu
)

val_data_loader = torch.utils.data.DataLoader(
    valid_dataset,
    shuffle = False, # we dont want to shuffle the valid dataset
    batch_size = VALID_BATCH_SIZE,
    num_workers = 0, # use all the gpu
)

In [26]:
device = torch('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

cpu


In [27]:
def load_ckp(checkpoint_fpath, model, optimizer):
  checkpoint = torch.load(checkpoint_fpath)
  model.load_state_dict(checkpoint('state_dict')) # loads the last state of the model
  optimizer.load_state_dict(checkpoint['optimizer'])
  valid_loss_min = checkpoint['valid_loss_min']
  return model, optimizer, checkpoint['epoch'], valid_loss_min.item()

def save_ckp(state, is_best, checkpoint_path, best_model_path):
  f_path = checkpoint_path
  torch.save(state, f_path)
  if is_best:
    best_fpath = best_model_path
    shutil.copyfile(f_path, best_fpath)

In [28]:
class BERTClass(torch.nn.Module):
  def __init__(self):
    super(BERTClass, self).__init__()
    self.bert_model = BertModel.from_pretrained('bert-base-uncased', return_dict=True)
    self.dropout = torch.nn.Dropout(0.3)
    self.linear = torch.nn.Linear(768, 6)

  def forward(self, input_ids, attn_mask, token_type_ids):
    output = self.bert_model(
        input_ids,
        attention_mask = attn_mask,
        token_type_ids = token_type_ids
    )
    output_dropout = self.dropout(output.pooler_output)
    output = self.linear(output_dropout)
    return output

model = BERTClass()
model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERTClass(
  (bert_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [29]:
# Defining the Loss Function
def loss_fn(outputs, targets):
  return torch.nn.BCEWithLogitsLoss()(outputs, targets)

optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [30]:
val_targets=[]
val_outputs=[]

In [31]:
# Creating the Trainig Loop
def train_model(
    n_epochs,
    training_loader,
    validation_loader,
    model,
    optimizer,
    checkpoint_path,
    best_model_path
):
  valid_loss_min = np.Inf

  for epoch in range(1, n_epochs + 1):
    train_loss = 0
    valid_loss = 0

    model.train()
    print('############# Epoch {}: Training Start   #############'.format(epoch))

    # training loop
    for batch_idx, data in enumerate(training_loader):
      ids = data['input_ids'].to(device, dtype=torch.long)
      mask = data['attention_mask'].to(device, dtype=torch.long)
      token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
      targets = data['targets'].to(device, dtype=torch.float)

      outputs = model(ids, mask, token_type_ids)

      optimizer.zero_grad()
      loss = loss_fn(outputs, targets)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      train_loss = train_loss + ((1/(batch_idx + 1))*(loss.item()-train_loss))

    print('############# Epoch {}: Training End     #############'.format(epoch))
    print('############# Epoch {}: Validation Start   #############'.format(epoch))

    # validate the model
    model.eval()

    with torch.no_grad():
      for batch_idx, data in enumerate(validation_loader, 0):
        ids = data['input_ids'].to(device, dtype=torch.long)
        mask = data['attention_mask'].to(device, dtype=torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
        targets = data['targets'].to(device, dtype=torch.float)
        output = model(ids, mask, token_type_ids)

        loss = loss_fn(outputs, targets)
        valid_loss = valid_loss + ((1/(batch_idx+1))*(loss.item()-valid_loss))
        valid_loss = valid_loss + ((1 / (batch_idx + 1)) * (loss.item() - valid_loss))

    print('############# Epoch {}: Validation End     #############'.format(epoch))
    # calculate average losses
    #print('before cal avg train loss', train_loss)
    train_loss = train_loss/len(training_loader)
    valid_loss = valid_loss/len(validation_loader)
    # print training/validation statistics
    print('Epoch: {} \tAvgerage Training Loss: {:.6f} \tAverage Validation Loss: {:.6f}'.format(
          epoch,
          train_loss,
          valid_loss
          ))

    # create checkpoint variable and add important data
    checkpoint = {
        'epoch': epoch + 1,
        'valid_loss_min': valid_loss,
        'state_dict':model.state_dict(),
        'optimizer': optimizer.state_dict()
    }

    # save checkpoint
    save_ckp(checkpoint, False, checkpoint_path, best_model_path)

    ## TODO: save the model if validation loss has decreased
    if valid_loss <= valid_loss_min:
      print('Validation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min,valid_loss))
      # save checkpoint as best model
      save_ckp(checkpoint, True, checkpoint_path, best_model_path)
      valid_loss_min = valid_loss

  return model

In [32]:
ckpt_path = "/content/drive/MyDrive/datasets/multi-label/curr_ckpt"
best_model_path = "/content/drive/MyDrive/datasets/multi-label/best_model.pt"

In [ ]:
trained_model = train_model(EPOCHS, train_data_loader, val_data_loader, model, optimizer, "/curr_ckpt", '/best.pt')

############# Epoch 1: Training Start   #############
